In [6]:
import pandas as pd
import numpy as np


#Pt1.Ingesta tecnica:
print("Iniciando carga y limpieza de datos...")
try:
    df = pd.read_csv('12_bmw_data.csv')
    df = df.dropna(subset=['Date', 'Adj_Close', 'Volume'])
        
except FileNotFoundError:
    print("Error: El archivo '12_bmw_data.csv' no se encuentra en el directorio.")

#Limpieza de Adj_close
df['Adj_Close'] = df['Adj_Close'].astype(str)
df['Adj_Close'] = df['Adj_Close'].str.replace('$', '', regex=False)
df['Adj_Close'] = df['Adj_Close'].str.replace('€', '', regex=False)
df['Adj_Close'] = df['Adj_Close'].str.replace(',', '', regex=False)
df['Adj_Close'] = pd.to_numeric(df['Adj_Close'], errors='coerce')

#Modificar valores nulos con la mediana
mediana_precio = df['Adj_Close'].median()
df['Adj_Close'] = df['Adj_Close'].fillna(mediana_precio)

#Conversion de fechas
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df = df.sort_values('Date')

Iniciando carga y limpieza de datos...


In [7]:
#Pt2.Ingenieria de caracteristicas:
  
df['daily_range'] = df['High'] - df['Low'] #Rango diario
df['daily_return'] = df['Adj_Close'].pct_change() * 100 #Rendimiento diario
df['year'] = df['Date'].dt.year #Año



# Pt3.Filtrado cruzado:
print("Aplicando filtros de volumen de ventas...")

# Condición 1: Volumen superior a la media
cond_volumen = df['Volume'] > df['Volume'].mean()
    
# Condición 2: Volatilidad alta (Días de movimiento significativo)
cond_volatilidad = df['daily_range'] > df['daily_range'].median()
    
# Extraemos el subconjunto crítico (df_target) con visibilidad condicionada
df_target = df[cond_volumen & cond_volatilidad][['Date', 'Adj_Close', 'daily_return']].copy()

print(f"Registros en df_target que cumplen los criterios lógicos: {len(df_target)}")



Aplicando filtros de volumen de ventas...
Registros en df_target que cumplen los criterios lógicos: 1560


In [8]:
#Pt4.Agregacion y analisis:
print("Agrupando datos por año...")

df_resumen_anual = df.groupby('year').agg(
    precio_medio_cierre=('Adj_Close', 'mean'),       
    maximo_historico_anual=('High', 'max'),            
    volumen_total_anual=('Volume', 'sum'),             
    volatilidad_media_anual=('daily_range', 'mean'),   
    num_sesiones=('Date', 'count')                     
    )

# Ordenamos por volumen total de mayor a menor
df_resumen_anual = df_resumen_anual.sort_values(by='volumen_total_anual', ascending=False)



#Pt5.Exportacion de resultados:
print("\n--- TOP 10 REGISTROS DE ALTA VOLATILIDAD  ---")
print(df_target.head(10).to_string())

# Exportamos separando por tabulador
print("\nExportando reporte anual a CSV...")
df_resumen_anual.to_csv('reporte_anual_bmw.csv', sep='\t', index=True)
print("Proceso finalizado con éxito. Archivo: 'reporte_anual_bmw.csv' generado.")

print("\n ", df_resumen_anual)


Agrupando datos por año...

--- TOP 10 REGISTROS DE ALTA VOLATILIDAD  ---
          Date  Adj_Close  daily_return
98  1997-03-26  11.633107      5.947798
114 1997-04-17  12.334319     -1.505773
207 1997-08-26  11.396399     -3.272774
208 1997-08-27  11.527905      1.153917
217 1997-09-09  11.974574      2.550187
229 1997-09-25  12.566574      1.450299
230 1997-09-26  13.070309      4.008528
252 1997-10-28  10.475857     -7.004371
253 1997-10-29  11.026844      5.259591
254 1997-10-30  10.870377     -1.418968

Exportando reporte anual a CSV...
Proceso finalizado con éxito. Archivo: 'reporte_anual_bmw.csv' generado.

        precio_medio_cierre  maximo_historico_anual  volumen_total_anual  \
year                                                                     
2008            15.498516               43.619999           1226378296   
2007            22.458710               51.490002            962945586   
2011            31.262118               73.849998            830265480   
2009 